# Multimodal Short-Video Recommendation System

This project aims to build a recommendation system to improve content relevance on a mobile short-video platform. The system predicts user engagement with candidate videos, measured by watch ratio, which is then used to rank candidate videos and recommend the most relevant content to users.

In [1]:
# Install emoji library if not present
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 12.7 MB/s eta 0:00:00


In [2]:
# Import libraries
import os
import re
import numpy as np
import pandas as pd
import joblib
import torch
import emoji as emoji_lib
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE      = "/content/drive/MyDrive/BT4222 Data/"
ARTIFACTS = BASE + "artifacts/"
os.makedirs(ARTIFACTS, exist_ok=True)

Mounted at /content/drive


## Step 3: Data Preprocessing & Feature Engineering

This notebook processes raw short-video interaction data into model-ready artifacts for predicting `watch_ratio`.

**What is produced and used where:**

| Artifact | Used in |
|---|---|
| `train/val/test_engineered.csv` | Notebook 4 (caption filtering), Notebook 5 (NeuMF) |
| `train/val/test_scaled.csv` | Notebook 5 (NeuMF training) |
| `user_continuous_matrix.npy` | Notebook 5 — Model B/C/D static user features |
| `video_continuous_matrix.npy` | Notebook 5 — Model B/C/D static video features |
| `user_fre_city_encoded.npy` | Notebook 5 — trainable city embedding |
| `video_root_id_encoded.npy` | Notebook 5 — trainable root category embedding |
| `pca_title_cn_128.npy` | Notebook 4 (caption filtering), Notebook 5 (Model B/C/D, concat at training time) |
| `pca_tags_cn_64.npy` | Notebook 4 (caption filtering), Notebook 5 (Model B/C/D, concat at training time) |
| `pca_title_en_128.npy` | Notebook 4 (caption filtering) only |
| `pca_asr_en_128.npy` | Notebook 4 (caption filtering) only |
| `model_meta.pkl` | Notebooks 4 & 5 — dimensions and config |
| Scalers (`interaction/user/video_scaler.pkl`) | Notebook 5 — inference-time scaling |

**Model feature plan (Notebook 5):**

| Model | User side | Video side | Interaction |
|---|---|---|---|
| **A — ID only** | `nn.Embedding(user_id)` | `nn.Embedding(item_id)` | — |
| **B — Static** | A + continuous matrix + `fre_city` embedding | A + continuous matrix + `root_id` embedding + title_cn PCA + tags_cn PCA | — |
| **C — Full context** | B | B | dynamic history + temporal features |
| **Hybrid** | Best of A/B/C | Best of A/B/C | Best of A/B/C |

### 3.1 Entity Extraction

1. Load and clean raw interaction data (0-indexed PID/user encoding)
2. Extract distinct video, user, and behavior attribute tables
3. Validate referential integrity and export

#### 3.1.1 Load and Clean Raw Interaction Data

In [4]:
df = pd.read_csv(BASE + 'interaction.csv')

print(f"Original shape: {df.shape}")
print(f"Unique interactions (user_id, pid, exposed_time): "
      f"{df.drop_duplicates(['user_id', 'pid', 'exposed_time']).shape[0]:,}")

# ── PID encoding (0-indexed contiguous integers for nn.Embedding) ──
unique_pids_sorted = sorted(df['pid'].unique())
pid_label_map = {pid: idx for idx, pid in enumerate(unique_pids_sorted)}
df['pid'] = df['pid'].map(pid_label_map)
assert df['pid'].isna().sum() == 0
joblib.dump(pid_label_map, ARTIFACTS + 'pid_label_map.pkl')
print(f"PIDs: {df['pid'].nunique():,} unique  [{df['pid'].min()}–{df['pid'].max()}]  ✓")

# ── User ID encoding ──
unique_users_sorted = sorted(df['user_id'].unique())
user_label_map = {uid: idx for idx, uid in enumerate(unique_users_sorted)}
df['user_id'] = df['user_id'].map(user_label_map)
assert df['user_id'].isna().sum() == 0
joblib.dump(user_label_map, ARTIFACTS + 'user_label_map.pkl')
print(f"Users: {df['user_id'].nunique():,} unique  [{df['user_id'].min()}–{df['user_id'].max()}]  ✓")

Original shape: (6767010, 28)
Unique interactions (user_id, pid, exposed_time): 1,019,568
PIDs: 153,561 unique  [0–153560]  ✓
Users: 10,000 unique  [0–9999]  ✓


#### 3.1.2 Entity Extraction: Video Data

Aggregate tags into a pipe-delimited string per video and extract the three-level category hierarchy. Only `root_id` will be used as a trainable embedding — `parent_id` and `category_id` are kept temporarily for upward imputation only.

In [5]:
# ── Aggregate tags per video ──
tags_per_video = (
    df.groupby('pid')['tag_name']
    .apply(lambda x: '|'.join(x.dropna().unique()))
    .reset_index()
    .rename(columns={'tag_name': 'tags'})
)

# ── Extract hierarchical category IDs (needed only for root imputation) ──
root_id = (
    df[df['category_level'] == 1][['pid', 'category_id']]
    .drop_duplicates('pid').rename(columns={'category_id': 'root_id'})
)
parent_id = (
    df[df['category_level'] == 2][['pid', 'category_id']]
    .drop_duplicates('pid').rename(columns={'category_id': 'parent_id'})
)
category_id = (
    df[df['category_level'] == 3][['pid', 'category_id']]
    .drop_duplicates('pid').rename(columns={'category_id': 'category_id'})
)

# ── Base video metadata ──
video_base = (
    df[['pid', 'author_fans_count', 'duration', 'title']]
    .drop_duplicates('pid').reset_index(drop=True)
)

video_data = (
    video_base
    .merge(tags_per_video, on='pid', how='left')
    .merge(root_id,   on='pid', how='left')
    .merge(parent_id, on='pid', how='left')
    .merge(category_id, on='pid', how='left')
)

print(f"Video data shape: {video_data.shape}")
print(f"Unique videos: {video_data['pid'].nunique():,}")
video_data.head(3)

Video data shape: (153561, 8)
Unique videos: 153,561


,pid,author_fans_count,duration,title,tags,root_id,parent_id,category_id
0,97473,46761,91.9,现如今为啥,正能量|电视机|内容过于真实,36.0,373.0,2315.0
1,17717,21610,102.8,择在韩国生活的原因之一吧,越努力越幸运加油|记录韩国打工生活|中国人在韩国奋斗,11.0,378.0,NaN
2,36547,98409,151.4,年轻人送外卖?还是进工厂?,商业|职场,22.0,NaN,NaN


#### 3.1.3 Entity Extraction: User Data

In [6]:
user_cols = ['user_id', 'gender', 'age', 'mod_price',
             'fre_city', 'fre_community_type', 'fre_city_level']

user_data = (
    df[user_cols]
    .drop_duplicates('user_id')
    .reset_index(drop=True)
)

print(f"User data shape: {user_data.shape}")
print(f"Unique users: {user_data['user_id'].nunique():,}")

User data shape: (10000, 7)
Unique users: 10,000


#### 3.1.4 Entity Extraction: Behavior Data

In [7]:
behavior_cols = [
    'user_id', 'pid', 'exposed_time', 'p_date', 'p_hour',
    'watch_time', 'cvm_like', 'click', 'comment',
    'follow', 'collect', 'forward', 'hate'
]

behavior_data = (
    df[behavior_cols]
    .drop_duplicates(['user_id', 'pid', 'exposed_time'])
    .reset_index(drop=True)
)

print(f"Behavior data shape: {behavior_data.shape}")
print(f"Unique interactions: {len(behavior_data):,}")

Behavior data shape: (1019568, 13)
Unique interactions: 1,019,568


#### 3.1.5 Validation & Export

In [8]:
assert user_data['user_id'].nunique() == len(user_data),  "Duplicate users"
assert video_data['pid'].nunique()   == len(video_data),  "Duplicate videos"
assert len(set(behavior_data['pid'])    - set(video_data['pid']))    == 0, "PIDs missing from video_data"
assert len(set(behavior_data['user_id']) - set(user_data['user_id'])) == 0, "Users missing from user_data"
print("✓ Referential integrity OK")
print(f"  Behavior: {len(behavior_data):,} rows  |  Videos: {len(video_data):,}  |  Users: {len(user_data):,}")

behavior_data.to_csv(BASE + 'behavior_data.csv', index=False)
user_data.to_csv(BASE + 'user_data.csv', index=False)
video_data.to_csv(BASE + 'video_data.csv', index=False)
print("✓ Exported behavior_data.csv, user_data.csv, video_data.csv")

✓ Referential integrity OK
  Behavior: 1,019,568 rows  |  Videos: 153,561  |  Users: 10,000
✓ Exported behavior_data.csv, user_data.csv, video_data.csv


In [9]:
# ── Reload from disk (makes section 3 re-runnable independently) ──
behavior_df  = pd.read_csv(BASE + 'behavior_data.csv')
video_df     = pd.read_csv(BASE + 'video_data.csv')
user_df      = pd.read_csv(BASE + 'user_data.csv')
category_df  = pd.read_csv(BASE + 'categories_cn_en.csv')

binary_cols = ['cvm_like', 'click', 'comment', 'follow', 'collect', 'forward', 'hate']
for col in binary_cols:
    behavior_df[col] = behavior_df[col].map(
        {'True': 1, 'False': 0, True: 1, False: 0}).fillna(0).astype(int)
behavior_df['watch_time'] = pd.to_numeric(behavior_df['watch_time'], errors='coerce').fillna(0)

print(f"Behavior: {behavior_df.shape}  |  Video: {video_df.shape}  |  User: {user_df.shape}")

Behavior: (1019568, 13)  |  Video: (153561, 8)  |  User: (10000, 7)


### 3.2 Target Variable Creation

`watch_ratio = watch_time / duration`, clipped to **[0, 2]**.

- Values > 1 indicate replay (strong engagement signal, ~29% of interactions).
- Clipping at 2 preserves the replay signal while limiting extreme outlier influence.
- Raw ratio is used as the MSE regression target (no log-transform needed).

In [10]:
behavior_df = behavior_df.merge(video_df[['pid', 'duration']], on='pid', how='left')

behavior_df['duration']    = behavior_df['duration'].replace(0, np.nan)
behavior_df['watch_ratio'] = (
    (behavior_df['watch_time'] / behavior_df['duration'])
    .fillna(0).clip(0, 2)
)

print(f"watch_ratio distribution:")
print(behavior_df['watch_ratio'].describe().round(4))
print(f"Replay rate (watch_ratio > 1): {(behavior_df['watch_ratio'] > 1).mean():.1%}")

watch_ratio distribution:
count    1.019568e+06
mean     5.724000e-01
std      5.893000e-01
min      0.000000e+00
25%      5.730000e-02
50%      3.224000e-01
75%      1.020900e+00
max      2.000000e+00
Name: watch_ratio, dtype: float64
Replay rate (watch_ratio > 1): 28.9%


### 3.3 Dynamic Feature Engineering

Dynamic features are **interaction-level** — they change per row.

- **Temporal context (3.3.1):** row-level transforms on `p_hour` / `p_date`.
- **Point-in-time history (3.3.2, 3.3.3):** `shift(1).expanding().mean()` so each row only sees strictly prior interactions. No leakage regardless of split.

#### 3.3.1 Temporal Context Features

In [11]:
behavior_df['p_date_dt'] = pd.to_datetime(behavior_df['p_date'].astype(str), format='%Y%m%d')

behavior_df['time_of_day_sin']  = np.sin(2 * np.pi * behavior_df['p_hour'] / 24)
behavior_df['time_of_day_cos']  = np.cos(2 * np.pi * behavior_df['p_hour'] / 24)
behavior_df['day_of_week']      = behavior_df['p_date_dt'].dt.dayofweek
behavior_df['day_of_week_sin']  = np.sin(2 * np.pi * behavior_df['day_of_week'] / 7)
behavior_df['day_of_week_cos']  = np.cos(2 * np.pi * behavior_df['day_of_week'] / 7)
behavior_df['is_weekend']       = (behavior_df['day_of_week'] >= 5).astype(int)

print("✓ Temporal features added:",
      ['time_of_day_sin', 'time_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend'])

✓ Temporal features added: ['time_of_day_sin', 'time_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend']


#### 3.3.2 Dynamic User Features (Point-in-Time Historical Means)

For each interaction: *"What is this user's average behaviour across all prior interactions?"*

`user_is_cold = 1` on a user's very first interaction (cold-start flag).

In [12]:
behavior_df = behavior_df.sort_values(['user_id', 'exposed_time']).reset_index(drop=True)

def get_historical_mean(df, group_col, target_col):
    """Expanding mean of strictly prior rows (shift(1) excludes current row)."""
    shifted = df.groupby(group_col)[target_col].shift(1)
    cumsum  = shifted.groupby(df[group_col]).cumsum()
    count   = shifted.groupby(df[group_col]).cumcount()
    return (cumsum / count).fillna(0)

user_engagement_cols = {
    'user_hist_watch_ratio':  'watch_ratio',
    'user_hist_click_rate':   'click',
    'user_hist_like_rate':    'cvm_like',
    'user_hist_comment_rate': 'comment',
    'user_hist_follow_rate':  'follow',
    'user_hist_collect_rate': 'collect',
    'user_hist_forward_rate': 'forward',
    'user_hist_hate_rate':    'hate',
}
for feat, col in user_engagement_cols.items():
    behavior_df[feat] = get_historical_mean(behavior_df, 'user_id', col)

behavior_df['user_is_cold'] = (behavior_df.groupby('user_id').cumcount() == 0).astype(int)

print(f"✓ User dynamic features: {list(user_engagement_cols)} + user_is_cold")
print(f"  Cold-start rows: {behavior_df['user_is_cold'].sum():,}")

✓ User dynamic features: ['user_hist_watch_ratio', 'user_hist_click_rate', 'user_hist_like_rate', 'user_hist_comment_rate', 'user_hist_follow_rate', 'user_hist_collect_rate', 'user_hist_forward_rate', 'user_hist_hate_rate'] + user_is_cold
  Cold-start rows: 10,000


#### 3.3.3 Dynamic Video Features (Point-in-Time Historical Means)

Mirror of 3.3.2, grouped by `pid`. Answers: *"How has this video performed across all prior viewers?"*

`video_hist_exposures` = log1p(cumulative prior view count).

In [13]:
behavior_df = behavior_df.sort_values(['pid', 'exposed_time']).reset_index(drop=True)

behavior_df['video_hist_exposures'] = np.log1p(behavior_df.groupby('pid').cumcount())

video_engagement_cols = {
    'video_hist_watch_ratio':  'watch_ratio',
    'video_hist_click_rate':   'click',
    'video_hist_like_rate':    'cvm_like',
    'video_hist_comment_rate': 'comment',
    'video_hist_follow_rate':  'follow',
    'video_hist_collect_rate': 'collect',
    'video_hist_forward_rate': 'forward',
    'video_hist_hate_rate':    'hate',
}
for feat, col in video_engagement_cols.items():
    behavior_df[feat] = get_historical_mean(behavior_df, 'pid', col)

behavior_df['video_is_cold'] = (behavior_df.groupby('pid').cumcount() == 0).astype(int)

print(f"✓ Video dynamic features: {list(video_engagement_cols)} + video_hist_exposures + video_is_cold")
print(f"  Cold-start rows: {behavior_df['video_is_cold'].sum():,}")

✓ Video dynamic features: ['video_hist_watch_ratio', 'video_hist_click_rate', 'video_hist_like_rate', 'video_hist_comment_rate', 'video_hist_follow_rate', 'video_hist_collect_rate', 'video_hist_forward_rate', 'video_hist_hate_rate'] + video_hist_exposures + video_is_cold
  Cold-start rows: 153,561


### 3.4 Temporal Train / Validation / Test Split

Chronological split — the model only learns from past behaviour to predict future engagement.

| Split | Dates | Purpose |
|---|---|---|
| Train | Sep 16–20 (5 days) | Model training |
| Validation | Sep 21 (1 day) | Hyperparameter tuning |
| Test | Sep 22 (1 day) | Final evaluation |

**Columns dropped before split:**
- **Leakage:** `watch_time`, `cvm_like`, `click`, `comment`, `follow`, `collect`, `forward`, `hate` — current-row outcomes; historical aggregates are retained.
- **Intermediate:** `p_date_dt`, `day_of_week`, `p_hour`, `duration` — already encoded or used to compute targets.

In [14]:
behavior_df = behavior_df.sort_values('exposed_time').reset_index(drop=True)

DROP_COLS = [
    # Leakage — current-row outcomes
    'watch_time', 'cvm_like', 'click', 'comment',
    'follow', 'collect', 'forward', 'hate',
    # Intermediate — served their purpose
    'p_date_dt', 'day_of_week', 'p_hour', 'duration',
]
behavior_engineered = behavior_df.drop(columns=DROP_COLS)

train_df = behavior_engineered[behavior_engineered['p_date'] <= 20220920]
val_df   = behavior_engineered[behavior_engineered['p_date'] == 20220921]
test_df  = behavior_engineered[behavior_engineered['p_date'] == 20220922]

total = len(behavior_engineered)
print(f"Train: {len(train_df):,} ({len(train_df)/total:.1%})  "      f"Val: {len(val_df):,} ({len(val_df)/total:.1%})  "      f"Test: {len(test_df):,} ({len(test_df)/total:.1%})")

train_df.to_csv(BASE + 'train_engineered.csv', index=False)
val_df.to_csv(BASE + 'val_engineered.csv', index=False)
test_df.to_csv(BASE + 'test_engineered.csv', index=False)
print("✓ Exported train/val/test_engineered.csv")
print(f"  Columns: {behavior_engineered.columns.tolist()}")

Train: 800,827 (78.5%)  Val: 128,091 (12.6%)  Test: 90,650 (8.9%)
✓ Exported train/val/test_engineered.csv
  Columns: ['user_id', 'pid', 'exposed_time', 'p_date', 'watch_ratio', 'time_of_day_sin', 'time_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend', 'user_hist_watch_ratio', 'user_hist_click_rate', 'user_hist_like_rate', 'user_hist_comment_rate', 'user_hist_follow_rate', 'user_hist_collect_rate', 'user_hist_forward_rate', 'user_hist_hate_rate', 'user_is_cold', 'video_hist_exposures', 'video_hist_watch_ratio', 'video_hist_click_rate', 'video_hist_like_rate', 'video_hist_comment_rate', 'video_hist_follow_rate', 'video_hist_collect_rate', 'video_hist_forward_rate', 'video_hist_hate_rate', 'video_is_cold']


### 3.5 Static Feature Engineering

Static features are fixed per user or per video — they do not change between interactions.

#### 3.5.1 User Features

| Feature | Type | Encoding |
|---|---|---|
| `gender`, `fre_community_type`, `fre_city_level`, `age_group` | Low-cardinality categorical | One-hot |
| `fre_city` | High-cardinality categorical | Label-encoded → `nn.Embedding` (trainable) |
| `mod_price` | Continuous, right-skewed | log1p → StandardScaler |
| `user_total_interactions_log`, `user_avg_watch_ratio` | Training-set priors | StandardScaler |

In [15]:
# 1. Age binning
age_bins   = [0, 18, 25, 35, 45, 55, 120]
age_labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55+']
user_df['age_group'] = pd.cut(
    user_df['age'], bins=age_bins, labels=age_labels, right=False).astype(str)
user_df = user_df.drop(columns=['age'])

# 2. One-hot encode low-cardinality categoricals
low_card_cols = ['gender', 'fre_community_type', 'fre_city_level', 'age_group']
user_df = pd.get_dummies(user_df, columns=low_card_cols, drop_first=False, dtype=int)
print(f"  One-hot → {user_df.shape[1]} columns")

# 3. fre_city → label-encoded for nn.Embedding (trainable in NeuMF)
city_counts = user_df['fre_city'].value_counts()
rare_cities = city_counts[city_counts < 10].index
user_df['fre_city'] = user_df['fre_city'].replace(rare_cities, 'other')
print(f"  Grouped {len(rare_cities)} rare cities into 'other'")

le_city = LabelEncoder()
user_df['fre_city_encoded'] = le_city.fit_transform(user_df['fre_city'])
print(f"  fre_city: {len(le_city.classes_)} classes → [0, {len(le_city.classes_)-1}]")
joblib.dump(le_city, ARTIFACTS + 'le_city.pkl')

# 4. Log-transform phone price
user_df['mod_price_log'] = np.log1p(user_df['mod_price'])

# 5. Drop raw columns
user_df = user_df.drop(columns=['fre_city', 'mod_price'])

print(f"✓ User features engineered. Shape: {user_df.shape}")

  One-hot → 22 columns
  Grouped 99 rare cities into 'other'
  fre_city: 275 classes → [0, 274]
✓ User features engineered. Shape: (10000, 22)


#### 3.5.2 Video Features

**Processing steps:**
1. Log-transform `duration` and `author_fans_count`
2. Impute `root_id` upward from `parent_id` → `category_id`, then label-encode for `nn.Embedding`
3. Drop `parent_id` and `category_id` — only `root_id` is used
4. Count and log-transform tags (`num_tags_log`)
5. Load `title_en` and `asr_en`, clean for BERT
6. BERT + PCA for: `title_cn` (128-d), `tags_cn` (64-d), `title_en` (128-d), `asr_en` (128-d)

**Note on text embeddings:**
- `title_cn` (128-d) and `tags_cn` (64-d)
- `title_en` (128-d) and `asr_en` (128-d)
- None of the text embeddings are baked into `video_continuous_matrix` — they are kept as separate `.npy` files for flexibility.

In [16]:
video_df = video_df.sort_values('pid').reset_index(drop=True)

# ── 1. Log-transform continuous features ──
video_df['duration_log']    = np.log1p(video_df['duration'])
video_df['author_fans_log'] = np.log1p(video_df['author_fans_count'])

# ── 2. Category hierarchy: impute root_id, label-encode, drop parent/category ──
# Build lookup dictionaries from category taxonomy
cat_only      = category_df[category_df['category_level'] == 3]
cat_to_parent = dict(zip(cat_only['category_id'], cat_only['parent_id']))
cat_to_root   = dict(zip(cat_only['category_id'], cat_only['root_id']))
parents_only  = category_df[category_df['category_level'] == 2]
parent_to_root = dict(zip(parents_only['category_id'], parents_only['root_id']))

# Upward imputation: fill root_id from finer-grained levels where missing
video_df['parent_id'] = video_df['parent_id'].fillna(
    video_df['category_id'].map(cat_to_parent))
video_df['root_id'] = video_df['root_id'].fillna(
    video_df['parent_id'].map(parent_to_root))
video_df['root_id'] = video_df['root_id'].fillna(
    video_df['category_id'].map(cat_to_root))
video_df['root_id'] = video_df['root_id'].fillna(-1).astype(int)

# Label-encode root_id for nn.Embedding (trainable in NeuMF)
le_root = LabelEncoder()
video_df['root_id_encoded'] = le_root.fit_transform(video_df['root_id'])
print(f"  root_id: {len(le_root.classes_)} classes → [0, {len(le_root.classes_)-1}]")
joblib.dump(le_root, ARTIFACTS + 'le_root.pkl')

# Drop parent_id and category_id — only needed for root imputation
video_df = video_df.drop(columns=['parent_id', 'category_id', 'root_id'])

print(f"✓ Category imputed and encoded. Dropped parent_id, category_id.")

# ── 3. Tags: scalar count only (BERT below) ──
video_df['tags'] = video_df['tags'].fillna('无标签')
video_df['num_tags']     = video_df['tags'].apply(lambda x: len(x.split('|')))
video_df['num_tags_log'] = np.log1p(video_df['num_tags'])
print(f"  Tag counts: min={video_df['num_tags'].min()}, "      f"max={video_df['num_tags'].max()}, mean={video_df['num_tags'].mean():.1f}")

# ── 4. Load title_en and asr_en (guard merge against pid space mismatch) ──
video_full_raw = pd.read_csv(BASE + 'video_data_full.csv',
                              usecols=['pid', 'title_en', 'asr_en'])

# If video_data_full has raw pid while video_df has encoded pid,
# map raw -> encoded before merge.
overlap_before = video_df['pid'].isin(video_full_raw['pid']).mean()
print(f"  pid overlap before mapping: {overlap_before:.2%}")

if overlap_before < 0.90:
    pid_label_map = joblib.load(ARTIFACTS + 'pid_label_map.pkl')
    video_full_raw['pid'] = video_full_raw['pid'].map(pid_label_map)
    video_full_raw = video_full_raw.dropna(subset=['pid']).copy()
    video_full_raw['pid'] = video_full_raw['pid'].astype(int)

overlap_after = video_df['pid'].isin(video_full_raw['pid']).mean()
print(f"  pid overlap used for merge: {overlap_after:.2%}")
assert overlap_after > 0.90, "Low pid overlap after mapping; check pid encoding consistency."

video_full_raw = video_full_raw.drop_duplicates('pid')
video_df = video_df.merge(video_full_raw[['pid', 'title_en', 'asr_en']],
                           on='pid', how='left')
print(f"  title_en coverage: {video_df['title_en'].notna().sum():,} / {len(video_df):,}")
del video_full_raw

# ── 5. Clean EN text (emoji removal, whitespace normalisation) ──
def clean_en_text(text: str) -> str:
    text = emoji_lib.replace_emoji(text, replace=' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else 'no content'

video_df['title_en_clean'] = video_df['title_en'].fillna('no title').apply(clean_en_text)
video_df['asr_en_clean']   = video_df['asr_en'].fillna('no transcript').apply(clean_en_text)
print(f"  title_en emoji count: {video_df['title_en'].fillna('').apply(lambda x: bool(emoji_lib.emoji_count(x))).sum():,}")
print(f"  asr_en   emoji count: {video_df['asr_en'].fillna('').apply(lambda x: bool(emoji_lib.emoji_count(x))).sum():,}")
print("✓ Video metadata features prepared.")

  root_id: 36 classes → [0, 35]
✓ Category imputed and encoded. Dropped parent_id, category_id.
  Tag counts: min=1, max=16, mean=2.5
  pid overlap before mapping: 100.00%
  pid overlap used for merge: 100.00%
  title_en coverage: 153,557 / 153,561
  title_en emoji count: 0
  asr_en   emoji count: 18,256
✓ Video metadata features prepared.


#### 3.5.3 BERT + PCA Text Embeddings

All embeddings are extracted once and cached to disk. PCA is always fitted on training-set videos only to prevent leakage.

| Modality | Model | PCA dim
|---|---|---|
| `title_cn` | bert-base-chinese | 128 |
| `tags_cn` | bert-base-chinese | 64 |
| `title_en` | bert-base-uncased | 128 |
| `asr_en` | bert-base-uncased | 128 |

In [17]:
PCA_DIM_CN   = 128
PCA_DIM_EN   = 128
PCA_DIM_TAGS = 64

train_pids       = set(train_df['pid'].unique())
train_video_mask = video_df['pid'].isin(train_pids).values
print(f"Train videos for PCA: {train_video_mask.sum():,} / {len(video_df):,} ({train_video_mask.mean():.1%})")


def extract_bert_embeddings(texts, model_name, cache_path,
                             batch_size=256, max_length=128):
    """Extract [CLS] embeddings. Cached to disk — skips GPU on re-run."""
    if os.path.exists(cache_path):
        print(f"  Loaded from cache: {cache_path}")
        return np.load(cache_path)

    print(f"  Extracting with {model_name}...")
    _dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    _tok = AutoTokenizer.from_pretrained(model_name)
    _mod = AutoModel.from_pretrained(model_name).to(_dev)
    _mod.eval()

    all_cls = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=model_name.split('/')[-1]):
            batch  = texts[i: i + batch_size]
            inputs = _tok(batch, padding=True, truncation=True,
                          max_length=max_length, return_tensors='pt').to(_dev)
            out    = _mod(**inputs)
            all_cls.append(out.last_hidden_state[:, 0, :].cpu().numpy())

    del _mod, _tok
    torch.cuda.empty_cache()
    embs = np.vstack(all_cls)
    np.save(cache_path, embs)
    print(f"  Cached {embs.shape} → {cache_path}")
    return embs


def fit_pca(embeddings, name, n_components, train_mask):
    """Fit PCA on train videos only, transform all. Save PCA object."""
    pca = PCA(n_components=n_components, random_state=42)
    pca.fit(embeddings[train_mask])
    compressed = pca.transform(embeddings).astype(np.float32)
    joblib.dump(pca, ARTIFACTS + f'pca_{name}_{n_components}.pkl')
    print(f"  {name}: 768d → {n_components}d  "          f"(train var explained: {pca.explained_variance_ratio_.sum():.1%})")
    return compressed


# ── 7.1 Chinese Title ──
print("\n── title_cn (bert-base-chinese, 128-d) ──")
emb_cn  = extract_bert_embeddings(
    video_df['title'].fillna('无标题').tolist(),
    'bert-base-chinese', ARTIFACTS + 'emb_title_cn_768.npy', max_length=64)
pca_cn  = fit_pca(emb_cn,  'title_cn', PCA_DIM_CN,   train_video_mask)
np.save(ARTIFACTS + f'pca_title_cn_{PCA_DIM_CN}.npy', pca_cn)

# ── 7.2 Tags CN ──
print("\n── tags_cn (bert-base-chinese, 64-d) ──")
tags_text = video_df['tags'].apply(lambda x: x.replace('|', ' ')).tolist()
emb_tags  = extract_bert_embeddings(
    tags_text, 'bert-base-chinese',
    ARTIFACTS + 'emb_tags_cn_768.npy', max_length=64)
pca_tags  = fit_pca(emb_tags, 'tags_cn', PCA_DIM_TAGS, train_video_mask)
np.save(ARTIFACTS + f'pca_tags_cn_{PCA_DIM_TAGS}.npy', pca_tags)

# ── 7.3 English Title ──
print("\n── title_en (bert-base-uncased, 128-d) ──")
emb_en  = extract_bert_embeddings(
    video_df['title_en_clean'].tolist(),
    'bert-base-uncased', ARTIFACTS + 'emb_title_en_768.npy', max_length=64)
pca_en  = fit_pca(emb_en,  'title_en', PCA_DIM_EN,   train_video_mask)
np.save(ARTIFACTS + f'pca_title_en_{PCA_DIM_EN}.npy', pca_en)

# ── 7.4 ASR Transcript ──
print("\n── asr_en (bert-base-uncased, 128-d) ──")
emb_asr = extract_bert_embeddings(
    video_df['asr_en_clean'].tolist(),
    'bert-base-uncased', ARTIFACTS + 'emb_asr_en_768.npy', max_length=256)
pca_asr = fit_pca(emb_asr, 'asr_en',   PCA_DIM_EN,   train_video_mask)
np.save(ARTIFACTS + f'pca_asr_en_{PCA_DIM_EN}.npy', pca_asr)

print(f"""
✓ Text embeddings ready:
  pca_title_cn_{PCA_DIM_CN}.npy
  pca_tags_cn_{PCA_DIM_TAGS}.npy
  pca_title_en_{PCA_DIM_EN}.npy
  pca_asr_en_{PCA_DIM_EN}.npy
""")

# ── Drop raw/intermediate text columns from video_df ──
video_df = video_df.drop(columns=[
    'title', 'tags', 'title_en', 'asr_en',
    'title_en_clean', 'asr_en_clean', 'num_tags',
    'duration', 'author_fans_count',
], errors='ignore')
print(f"video_df remaining columns: {video_df.columns.tolist()}")

Train videos for PCA: 138,768 / 153,561 (90.4%)

── title_cn (bert-base-chinese, 128-d) ──
  Loaded from cache: /content/drive/MyDrive/BT4222 Data/artifacts/emb_title_cn_768.npy
  title_cn: 768d → 128d  (train var explained: 75.5%)

── tags_cn (bert-base-chinese, 64-d) ──
  Loaded from cache: /content/drive/MyDrive/BT4222 Data/artifacts/emb_tags_cn_768.npy
  tags_cn: 768d → 64d  (train var explained: 65.2%)

── title_en (bert-base-uncased, 128-d) ──
  Loaded from cache: /content/drive/MyDrive/BT4222 Data/artifacts/emb_title_en_768.npy
  title_en: 768d → 128d  (train var explained: 80.3%)

── asr_en (bert-base-uncased, 128-d) ──
  Loaded from cache: /content/drive/MyDrive/BT4222 Data/artifacts/emb_asr_en_768.npy
  asr_en: 768d → 128d  (train var explained: 85.4%)

✓ Text embeddings ready:
  pca_title_cn_128.npy
  pca_tags_cn_64.npy
  pca_title_en_128.npy
  pca_asr_en_128.npy

video_df remaining columns: ['pid', 'duration_log', 'author_fans_log', 'root_id_encoded', 'num_tags_log']


#### 3.5.4 Training-Set Aggregate Priors

Computed on training data only — no leakage into val/test. Users/videos with no training history receive neutral defaults (global mean watch_ratio / 0).

In [18]:
global_mean_wr = train_df['watch_ratio'].mean()
print(f"Global train mean watch_ratio: {global_mean_wr:.4f}")

# ── User priors ──
user_priors = train_df.groupby('user_id').agg(
    user_total_interactions=('watch_ratio', 'count'),
    user_avg_watch_ratio=('watch_ratio', 'mean'),
).reset_index()
user_priors['user_total_interactions_log'] = np.log1p(user_priors['user_total_interactions'])

user_df = user_df.drop(columns=[c for c in
    ['user_total_interactions_log', 'user_avg_watch_ratio'] if c in user_df.columns])
user_df = user_df.merge(
    user_priors[['user_id', 'user_total_interactions_log', 'user_avg_watch_ratio']],
    on='user_id', how='left')
user_df['user_total_interactions_log'] = user_df['user_total_interactions_log'].fillna(0)
user_df['user_avg_watch_ratio']        = user_df['user_avg_watch_ratio'].fillna(global_mean_wr)
print(f"  user_df: {user_df.shape}")

# ── Video priors ──
video_priors = train_df.groupby('pid').agg(
    video_total_exposures=('watch_ratio', 'count'),
    video_avg_watch_ratio=('watch_ratio', 'mean'),
).reset_index()
video_priors['video_total_exposures_log'] = np.log1p(video_priors['video_total_exposures'])

video_df = video_df.drop(columns=[c for c in
    ['video_total_exposures_log', 'video_avg_watch_ratio'] if c in video_df.columns])
video_df = video_df.merge(
    video_priors[['pid', 'video_total_exposures_log', 'video_avg_watch_ratio']],
    on='pid', how='left')
video_df['video_total_exposures_log'] = video_df['video_total_exposures_log'].fillna(0)
video_df['video_avg_watch_ratio']     = video_df['video_avg_watch_ratio'].fillna(global_mean_wr)
print(f"  video_df: {video_df.shape}")

Global train mean watch_ratio: 0.5782
  user_df: (10000, 24)
  video_df: (153561, 7)


### 3.6 Model-Ready Artifact Assembly

Separate each entity's features into:
- A **continuous feature matrix** (floats — fed into a linear projection in NeuMF)
- **Categorical ID arrays** (integers — fed into `nn.Embedding` layers)

Text embeddings (`pca_title_cn`, `pca_tags_cn`) are kept separate and concatenated in Notebook 5 per model tier.

#### 3.6.1 User Feature Matrix

In [19]:
# One-hot columns (already binary [0,1] — no scaling needed)
USER_ONEHOT_COLS = [c for c in user_df.columns
                    if c.startswith(('gender_', 'fre_community_type_',
                                     'fre_city_level_', 'age_group_'))]

# Continuous columns (3 scalar + one-hots)
USER_CONTINUOUS_COLS = (
    ['mod_price_log', 'user_total_interactions_log', 'user_avg_watch_ratio']
    + USER_ONEHOT_COLS
)

# Trainable embedding: fre_city
USER_EMBEDDING_COLS = {'fre_city_encoded': user_df['fre_city_encoded'].nunique()}

user_df_sorted = user_df.sort_values('user_id').reset_index(drop=True)
assert (user_df_sorted['user_id'] == np.arange(len(user_df_sorted))).all(),     "User IDs must be 0-indexed and contiguous!"

user_continuous_matrix = user_df_sorted[USER_CONTINUOUS_COLS].values.astype(np.float32)
user_cat_arrays = {col: user_df_sorted[col].values.astype(np.int64)
                   for col in USER_EMBEDDING_COLS}

print(f"User continuous matrix : {user_continuous_matrix.shape}")
print(f"  Scalar cols          : {['mod_price_log', 'user_total_interactions_log', 'user_avg_watch_ratio']}")
print(f"  One-hot cols         : {len(USER_ONEHOT_COLS)} columns")
print(f"  Trainable embedding  : fre_city_encoded ({USER_EMBEDDING_COLS['fre_city_encoded']} classes)")

User continuous matrix : (10000, 22)
  Scalar cols          : ['mod_price_log', 'user_total_interactions_log', 'user_avg_watch_ratio']
  One-hot cols         : 19 columns
  Trainable embedding  : fre_city_encoded (275 classes)


#### 3.6.2 Video Feature Matrix

In [20]:
# Continuous columns (scalar metadata only — text embeddings concatenated in NB5)
VIDEO_CONTINUOUS_COLS = [
    'author_fans_log',
    'duration_log',
    'num_tags_log',
    'video_total_exposures_log',
    'video_avg_watch_ratio',
]

# Trainable embedding: root_id (genre)
VIDEO_EMBEDDING_COLS = {'root_id_encoded': video_df['root_id_encoded'].nunique()}

video_df_sorted = video_df.sort_values('pid').reset_index(drop=True)
assert (video_df_sorted['pid'] == np.arange(len(video_df_sorted))).all(),     "PIDs must be 0-indexed and contiguous!"

video_continuous_matrix = video_df_sorted[VIDEO_CONTINUOUS_COLS].values.astype(np.float32)
video_cat_arrays = {col: video_df_sorted[col].values.astype(np.int64)
                    for col in VIDEO_EMBEDDING_COLS}

print(f"Video continuous matrix : {video_continuous_matrix.shape}")
print(f"  Cols                  : {VIDEO_CONTINUOUS_COLS}")
print(f"  Trainable embedding   : root_id_encoded ({VIDEO_EMBEDDING_COLS['root_id_encoded']} classes)")
print()
print("Text embeddings (NOT in matrix — loaded separately in NB5):")
print(f"  pca_title_cn_{PCA_DIM_CN}.npy  → concat for Model B/C/D")
print(f"  pca_tags_cn_{PCA_DIM_TAGS}.npy   → concat for Model B/C/D")

Video continuous matrix : (153561, 5)
  Cols                  : ['author_fans_log', 'duration_log', 'num_tags_log', 'video_total_exposures_log', 'video_avg_watch_ratio']
  Trainable embedding   : root_id_encoded (36 classes)

Text embeddings (NOT in matrix — loaded separately in NB5):
  pca_title_cn_128.npy  → concat for Model B/C/D
  pca_tags_cn_64.npy   → concat for Model B/C/D


#### 3.6.3 Interaction-Level Feature Columns

In [21]:
INTERACTION_FEATURE_COLS = [
    # Temporal
    'time_of_day_sin', 'time_of_day_cos',
    'day_of_week_sin', 'day_of_week_cos', 'is_weekend',
    # Dynamic user history
    'user_hist_watch_ratio', 'user_hist_click_rate', 'user_hist_like_rate',
    'user_hist_comment_rate', 'user_hist_follow_rate', 'user_hist_collect_rate',
    'user_hist_forward_rate', 'user_hist_hate_rate', 'user_is_cold',
    # Dynamic video history
    'video_hist_exposures', 'video_hist_watch_ratio', 'video_hist_click_rate',
    'video_hist_like_rate', 'video_hist_comment_rate', 'video_hist_follow_rate',
    'video_hist_collect_rate', 'video_hist_forward_rate', 'video_hist_hate_rate',
    'video_is_cold',
]

for col in INTERACTION_FEATURE_COLS:
    assert col in train_df.columns, f"Missing: {col}"

print(f"Interaction features: {len(INTERACTION_FEATURE_COLS)} dims")
print(f"  {INTERACTION_FEATURE_COLS}")

Interaction features: 24 dims
  ['time_of_day_sin', 'time_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'is_weekend', 'user_hist_watch_ratio', 'user_hist_click_rate', 'user_hist_like_rate', 'user_hist_comment_rate', 'user_hist_follow_rate', 'user_hist_collect_rate', 'user_hist_forward_rate', 'user_hist_hate_rate', 'user_is_cold', 'video_hist_exposures', 'video_hist_watch_ratio', 'video_hist_click_rate', 'video_hist_like_rate', 'video_hist_comment_rate', 'video_hist_follow_rate', 'video_hist_collect_rate', 'video_hist_forward_rate', 'video_hist_hate_rate', 'video_is_cold']


#### 3.6.4 Feature Scaling

StandardScaler fitted on training data only, applied to val/test.

- **Interaction features:** scale continuous columns; skip binary indicators and sin/cos (already bounded).
- **User matrix:** scale the 3 scalar columns only; one-hot columns are already in {0,1}.
- **Video matrix:** scale all 5 continuous columns.

In [22]:
# ── 1. Interaction features ──
INTERACTION_NO_SCALE = {
    'time_of_day_sin', 'time_of_day_cos', 'day_of_week_sin', 'day_of_week_cos',
    'is_weekend', 'user_is_cold', 'video_is_cold',
}
INTERACTION_SCALE_COLS = [c for c in INTERACTION_FEATURE_COLS if c not in INTERACTION_NO_SCALE]

assert len(train_df) > 0, "Train split is empty; cannot fit interaction scaler."

interaction_scaler = StandardScaler()
train_df[INTERACTION_SCALE_COLS] = interaction_scaler.fit_transform(train_df[INTERACTION_SCALE_COLS])
if len(val_df) > 0:
    val_df[INTERACTION_SCALE_COLS] = interaction_scaler.transform(val_df[INTERACTION_SCALE_COLS])
if len(test_df) > 0:
    test_df[INTERACTION_SCALE_COLS] = interaction_scaler.transform(test_df[INTERACTION_SCALE_COLS])

# ── 2. User matrix — scale scalar cols only ──
# Fit on train users only to avoid temporal leakage.
# `.unique()` ensures each user contributes once; without it, very active users
# appear many times and skew scaler statistics for a static entity matrix.
USER_SCALAR_COLS  = ['mod_price_log', 'user_total_interactions_log', 'user_avg_watch_ratio']
n_scalar = len(USER_SCALAR_COLS)
train_user_ids = np.sort(train_df['user_id'].unique())
assert train_user_ids.max() < len(user_continuous_matrix), "train user_id out of range"

user_scaler = StandardScaler()
user_scaler.fit(user_continuous_matrix[train_user_ids, :n_scalar])
user_continuous_matrix[:, :n_scalar] = user_scaler.transform(
    user_continuous_matrix[:, :n_scalar]
).astype(np.float32)

# ── 3. Video matrix — scale all ──
# Same logic as users: fit only on train videos, transform full matrix.
train_video_ids = np.sort(train_df['pid'].unique())
assert train_video_ids.max() < len(video_continuous_matrix), "train pid out of range"

video_scaler = StandardScaler()
video_scaler.fit(video_continuous_matrix[train_video_ids, :])
video_continuous_matrix = video_scaler.transform(video_continuous_matrix).astype(np.float32)

joblib.dump(interaction_scaler, ARTIFACTS + 'interaction_scaler.pkl')
joblib.dump(user_scaler,        ARTIFACTS + 'user_scaler.pkl')
joblib.dump(video_scaler,       ARTIFACTS + 'video_scaler.pkl')

print(f"✓ Interaction: {len(INTERACTION_SCALE_COLS)} scaled, {len(INTERACTION_NO_SCALE)} skipped")
print(f"✓ User matrix: {user_continuous_matrix.shape} (fit on {len(train_user_ids):,} unique train users)")
print(f"✓ Video matrix: {video_continuous_matrix.shape} (fit on {len(train_video_ids):,} unique train videos)")

/tmp/ipykernel_3440/1212881028.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[INTERACTION_SCALE_COLS] = interaction_scaler.fit_transform(train_df[INTERACTION_SCALE_COLS])
/tmp/ipykernel_3440/1212881028.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df[INTERACTION_SCALE_COLS] = interaction_scaler.transform(val_df[INTERACTION_SCALE_COLS])
/tmp/ipykernel_3440/1212881028.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

✓ Interaction: 17 scaled, 7 skipped
✓ User matrix: (10000, 22) (fit on 9,701 unique train users)
✓ Video matrix: (153561, 5) (fit on 138,768 unique train videos)


#### 3.6.5 Export All Artifacts

In [23]:
# ── Static matrices ──
np.save(ARTIFACTS + 'user_continuous_matrix.npy', user_continuous_matrix)
np.save(ARTIFACTS + 'video_continuous_matrix.npy', video_continuous_matrix)

for col, arr in user_cat_arrays.items():
    np.save(ARTIFACTS + f'user_{col}.npy', arr)
for col, arr in video_cat_arrays.items():
    np.save(ARTIFACTS + f'video_{col}.npy', arr)

# ── Scaled interaction splits ──
train_df.to_csv(BASE + 'train_scaled.csv', index=False)
val_df.to_csv(BASE + 'val_scaled.csv', index=False)
test_df.to_csv(BASE + 'test_scaled.csv', index=False)

# ── Engineered entity tables (for reference) ──
video_df_sorted.to_csv(BASE + 'video_data_engineered.csv', index=False)
user_df_sorted.to_csv(BASE + 'user_data_engineered.csv', index=False)

# ── Model metadata ──
model_meta = {
    # Counts
    'n_users': int(user_continuous_matrix.shape[0]),
    'n_items': int(video_continuous_matrix.shape[0]),
    # Dimensions
    'user_continuous_dim':  user_continuous_matrix.shape[1],
    'video_continuous_dim': video_continuous_matrix.shape[1],
    'interaction_dim':      len(INTERACTION_FEATURE_COLS),
    # Column configs
    'user_continuous_cols':     USER_CONTINUOUS_COLS,
    'video_continuous_cols':    VIDEO_CONTINUOUS_COLS,
    'user_embedding_cols':      USER_EMBEDDING_COLS,
    'video_embedding_cols':     VIDEO_EMBEDDING_COLS,
    'interaction_feature_cols': INTERACTION_FEATURE_COLS,
    # PCA dims
    'pca_dim_cn':   PCA_DIM_CN,    # 128 — title_cn
    'pca_dim_tags': PCA_DIM_TAGS,  # 64  — tags_cn
    'pca_dim_en':   PCA_DIM_EN,    # 128 — title_en, asr_en
    # Misc
    'global_mean_wr':  float(global_mean_wr),
    'text_modalities': ['title_cn', 'tags_cn', 'title_en', 'asr_en'],
}
joblib.dump(model_meta, ARTIFACTS + 'model_meta.pkl')

print("=" * 60)
print("  ARTIFACT EXPORT SUMMARY")
print("=" * 60)
print(f"  Users  : {model_meta['n_users']:,}")
print(f"    continuous matrix : {user_continuous_matrix.shape}  (scalar + one-hot)")
print(f"    trainable emb     : fre_city ({USER_EMBEDDING_COLS['fre_city_encoded']} classes)")
print()
print(f"  Videos : {model_meta['n_items']:,}")
print(f"    continuous matrix : {video_continuous_matrix.shape}  ({VIDEO_CONTINUOUS_COLS})")
print(f"    trainable emb     : root_id ({VIDEO_EMBEDDING_COLS['root_id_encoded']} classes)")
print(f"    text (NB5 B/C/D)  : title_cn {PCA_DIM_CN}d + tags_cn {PCA_DIM_TAGS}d (concat in NB5)")
print()
print(f"  Interaction features : {len(INTERACTION_FEATURE_COLS)}d (Model C)")
print(f"  Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(f"  Global mean watch_ratio : {global_mean_wr:.4f}")
print()
print(f"  Artifacts saved to: {ARTIFACTS}")
print("✓ All model-ready artifacts exported.")

  ARTIFACT EXPORT SUMMARY
  Users  : 10,000
    continuous matrix : (10000, 22)  (scalar + one-hot)
    trainable emb     : fre_city (275 classes)

  Videos : 153,561
    continuous matrix : (153561, 5)  (['author_fans_log', 'duration_log', 'num_tags_log', 'video_total_exposures_log', 'video_avg_watch_ratio'])
    trainable emb     : root_id (36 classes)
    text (NB5 B/C/D)  : title_cn 128d + tags_cn 64d (concat in NB5)

  Interaction features : 24d (Model C)
  Train: 800,827  Val: 128,091  Test: 90,650
  Global mean watch_ratio : 0.5782

  Artifacts saved to: /content/drive/MyDrive/BT4222 Data/artifacts/
✓ All model-ready artifacts exported.
